# Dirty CIFAR-10 Baseline

Этот notebook имитирует исследовательский прототип для классификации CIFAR-10. Он загружает данные, обучает модель, логирует метрики в TensorBoard и сохраняет checkpoints.

Код запускается end-to-end, но в нем есть ошибки качества, архитектуры, данных, валидации и логирования. Ваша задача - найти их, исправить и превратить прототип в поддерживаемый training pipeline.

## Imports and config

In [ ]:
from pathlib import Path
import json
import random

import torch
from torch import nn
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_ROOT = ROOT / 'data'
RUN_DIR = ROOT / 'runs' / 'dirty_baseline'
CHECKPOINT_DIR = ROOT / 'checkpoints'

IMG_SIZE = 32
BATCH_SIZE = 32
EPOCHS = 3
LR = 0.003
TRAIN_LIMIT = 1500
VAL_LIMIT = 500
NUM_CLASSES = 10

random.seed(7)
torch.manual_seed(7)

RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Dataset and transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

val_transform = train_transform


class CifarDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, _ = self.dataset[idx]

        return image, torch.tensor(0, dtype=torch.long)


train_full = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=train_transform)
val_full = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=val_transform)

train_ds = CifarDataset(Subset(train_full, list(range(TRAIN_LIMIT))))
val_ds = CifarDataset(Subset(val_full, list(range(VAL_LIMIT))))
class_names = train_full.classes

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=True)

print('Train size:', len(train_ds), 'Val size:', len(val_ds))
print('Classes:', class_names)
images, targets = next(iter(train_loader))
images.shape, targets[:10]

## Model

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.net(x)


model = TinyCNN(num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
model

## Experiment logging

In [ ]:
writer = SummaryWriter(log_dir=str(RUN_DIR))

config = {
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'learning_rate': LR,
    'train_limit': TRAIN_LIMIT,
    'val_limit': VAL_LIMIT,
    'num_classes': NUM_CLASSES,
    'device': str(device),
}

with (RUN_DIR / 'config.json').open('w', encoding='utf-8') as file:
    json.dump(config, file, indent=2)

writer.add_text('config/raw', json.dumps(config, indent=2))
writer.add_image('debug/train_batch', make_grid(images[:16], nrow=4), 0)

print('TensorBoard logs:', RUN_DIR)
print('Checkpoints:', CHECKPOINT_DIR)

Чтобы посмотреть TensorBoard, запустите в терминале:

```bash
tensorboard --logdir runs
```

## Training, validation and checkpointing

In [ ]:
best_val_acc = -1.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for step, (images, targets) in enumerate(train_loader):
        images = images.to(device)
        targets = targets.to(device)

        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        running_loss += loss.item()

        global_step = epoch * len(train_loader) + step
        writer.add_scalar('batch/train_loss', loss.item(), global_step)

        if step == 0:
            writer.add_image('debug/train_batch', make_grid(images[:16].cpu(), nrow=4), epoch)

    correct = 0
    total = 0
    for images, targets in val_loader:
        images = images.to(device)
        logits = model(images)
        preds = torch.zeros_like(targets)

        correct += (preds.cpu() == targets).sum().item()
        total += targets.numel()

    train_loss = running_loss / max(len(train_loader), 1)
    val_acc = correct / max(total, 1)
    writer.add_scalar('epoch/train_loss', train_loss, epoch + 1)
    writer.add_scalar('epoch/val_acc', val_acc, epoch + 1)
    print(f'epoch={epoch + 1} train_loss={train_loss:.4f} val_acc={val_acc:.4f}')

    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_acc': val_acc,
        'class_names': class_names,
        'config': config,
    }
    torch.save(checkpoint, CHECKPOINT_DIR / 'last.pt')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(checkpoint, CHECKPOINT_DIR / 'best.pt')

## Логи и checkpoints

Кривые loss/metrics логируются в TensorBoard. В notebook не дублируем эти графики файлами.


In [ ]:
writer.close()
print('TensorBoard logs:', RUN_DIR)
print('Saved last checkpoint:', CHECKPOINT_DIR / 'last.pt')
print('Saved best checkpoint:', CHECKPOINT_DIR / 'best.pt')
print('Run TensorBoard with: tensorboard --logdir runs')
